# Unit Tests — Reference

Run all 34 tests with:
```bash
venv/Scripts/python -m pytest tests/ -v
```

In [ ]:
import subprocess, sys, os

# Ensure the package is importable by pytest
env = os.environ.copy()
env["PYTHONPATH"] = os.path.abspath("..")

result = subprocess.run(
    [sys.executable, "-m", "pytest", "../tests/", "-v", "--tb=short"],
    capture_output=True, text=True, env=env
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

---

## `tests/test_config.py` — Configuration Validation

### `TestRegimeConfig`

| Test | What it verifies |
|------|------------------|
| `test_accepts_lists` | Accepts plain Python lists, converts to `float64` arrays |
| `test_validate_correct` | Correctly shaped scalar regime passes `validate(1)` |
| `test_validate_wrong_mu_shape` | Raises `ValueError` when `mu` has wrong length |
| `test_validate_wrong_K_shape` | Raises `ValueError` when `K` has wrong shape |
| `test_validate_wrong_sigma_shape` | Raises `ValueError` when `sigma` has wrong shape |

### `TestProcessConfig`

| Test | What it verifies |
|------|------------------|
| `test_valid_scalar_process` | Well-formed scalar process (d=1, 2 regimes) constructs OK |
| `test_valid_vector_process` | Well-formed vector process (d=2, 3 regimes) constructs OK |
| `test_no_regimes` | Empty `regimes` list raises `ValueError` |
| `test_transition_matrix_wrong_shape` | Shape mismatch with number of regimes raises `ValueError` |
| `test_transition_matrix_rows_not_summing_to_one` | Rows not summing to 1.0 raises `ValueError` |
| `test_transition_matrix_negative_entries` | Negative entries raise `ValueError` |
| `test_regime_dimension_mismatch` | K/mu/sigma dimensions not matching `d` raises `ValueError` |
| `test_x0_wrong_shape` | `x0` shape not matching `d` raises `ValueError` |
| `test_x0_valid` | Correctly shaped `x0` stored as-is |

### `TestStationaryDistribution`

| Test | What it verifies |
|------|------------------|
| `test_single_regime` | Single-regime process has $\pi = [1.0]$ |
| `test_symmetric_two_state` | Symmetric matrix yields uniform $\pi = [0.5, 0.5]$ |
| `test_asymmetric_two_state` | `[[0.9, 0.1], [0.3, 0.7]]` yields $\pi = [0.75, 0.25]$ |
| `test_three_state` | 3-state matrix satisfies $\pi P = \pi$, $\sum \pi = 1$, $\pi \geq 0$ |
| `test_sums_to_one` | Generic process's $\pi$ sums to 1 with non-negative entries |

---

## `tests/test_model.py` — Simulation Behaviour

### `TestScenarioModelInit`

| Test | What it verifies |
|------|------------------|
| `test_empty_processes` | Empty process list raises `ValueError` |
| `test_total_dim` | `total_dim` correctly sums dimensions (d=1 + d=2 = 3) |

### `TestSimulationShapes`

| Test | What it verifies |
|------|------------------|
| `test_single_scalar_process` | Single d=1 process: `values` $(M, T, 1)$, `regimes` $(M, T, 1)$ |
| `test_multi_process` | d=1 + d=2 model: `values` $(M, T, 3)$, `regimes` $(M, T, 2)$ |

### `TestReproducibility`

| Test | What it verifies |
|------|------------------|
| `test_same_seed_same_output` | Same seed produces bit-identical values and regimes |
| `test_different_seed_different_output` | Different seeds produce different output |

### `TestDeterministicProcess`

| Test | What it verifies |
|------|------------------|
| `test_sigma_zero_converges_to_mu` | $\Sigma = 0$, $K = 5$, $X_0 = 0$ converges to $\mu = 3$ after 500 steps at $dt = 0.01$ |

### `TestRegimeTransitions`

| Test | What it verifies |
|------|------------------|
| `test_absorbing_regime` | Identity transition matrix: each path stays in its initial regime |
| `test_regime_values_in_range` | All regime indices in $[0, R)$ |
| `test_transition_frequencies_approximate` | Over 5000 paths x 200 steps, observed frequencies match matrix within $\pm 0.02$ |

### `TestInitialConditions`

| Test | What it verifies |
|------|------------------|
| `test_default_x0_matches_sampled_regime_mu` | $X_0 = \mu$ of the sampled initial regime (two regimes with $\mu = \pm 10$) |
| `test_single_regime_x0_is_mu` | Single regime: all paths start at that regime's $\mu$ |
| `test_custom_x0` | User-provided `x0` overrides regime $\mu$ |
| `test_initial_regime_frequencies_match_stationary` | Over 10k paths, initial regime freq matches $\pi_0 = 0.75$ within $\pm 0.02$ |

### `TestVectorProcess`

| Test | What it verifies |
|------|------------------|
| `test_2d_deterministic_convergence` | 2D process with $\Sigma = 0$, $K = 3I$ converges from $(0, 0)$ to $\mu = (1, -2)$ |